In [ ]:
short_names = {
    'Pòlissa/Póliza/Policy': 'POLICY',
    'Tecnologia/Tecnología/Technology': 'TECHNOLOGY',
    'Diàmetre comptador (cm)/Diámetro contador (cm)/Counter diameter (cm)': 'DIAMETER',
    'Ús/Uso/Use': 'USAGE',
    "Tipus d'habitatge/Tipo de vivienda/Type of housing": 'HOUSING',
    'Data/Fecha/Date': 'HOUR/DATE',
    'Índex de lectura (L/h)/Índice de lectura (L/h)/Reading index (L/h)': 'CONSUMPTION',
}

KMEANS_THRESHHOLD = 2.5

#### Load data

In [ ]:
import pyarrow.dataset as ds
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
import pandas as pd

file_path = '../data/lectures_horaries_ABD.parquet'

dataset = ds.dataset(file_path, format="parquet")
table = dataset.to_table()
df = table.to_pandas()

In [ ]:
df = df.rename(columns=short_names)
# Convert 'Data/Fecha/Date' column to datetime format
df['HOUR/DATE'] = pd.to_datetime(df['HOUR/DATE'])

df['WEEKDAY'] = df['HOUR/DATE'].dt.dayofweek  # Extract weekday (Monday=0, Sunday=6)
df['HOUR'] = df['HOUR/DATE'].dt.hour

df = df.drop(columns=['HOUR/DATE'])

In [ ]:
df['FLOW'] = df.groupby('POLICY')['CONSUMPTION'].diff()
df = df.dropna(subset=["FLOW"])

In [ ]:
df

In [ ]:
negative_policies = df[df['FLOW'] < 0]['POLICY'].values
filtered_df = df[~df['POLICY'].isin(negative_policies)]

filtered_df

In [ ]:
filtered_df.loc[(filtered_df['USAGE'] == 'AJUNTAMENT') & (filtered_df['FLOW'] > 4000)] 

In [ ]:
ajuntament_flow_greater_than_4000 = filtered_df.loc[(filtered_df['USAGE'] == 'AJUNTAMENT') & (filtered_df['FLOW'] > 4000)]['POLICY'].values
new_filtered_df = filtered_df[~filtered_df['POLICY'].isin(ajuntament_flow_greater_than_4000)]

new_filtered_df

In [ ]:
new_filtered_df['WEEKDAY'].unique()

In [ ]:
print(len(new_filtered_df['POLICY'].unique()))

#### Calculate the mean flow for each hour, and plot it

In [ ]:
average_per_hour_usage = new_filtered_df.groupby(['USAGE', 'HOUR'])['FLOW'].mean().reset_index()

In [ ]:
# Assuming you have a DataFrame 'average_per_hour_usage' with columns 'HOUR', 'USAGE', and 'FLOW'
usage_types = average_per_hour_usage['USAGE'].unique()

fig, axs = plt.subplots(2, 2, figsize=(10, 8))

for i, usage in enumerate(usage_types):
  data = average_per_hour_usage[average_per_hour_usage['USAGE'] == usage]
  ax = axs[i // 2, i % 2]  # Assign subplot to each usage type
  ax.bar(data['HOUR'], data['FLOW'])  # Use bar function for bar graph
  ax.set_xlabel('Hour')
  ax.set_ylabel('Flow')
  ax.set_title(f'Flow for {usage}')

plt.tight_layout()
plt.show()

In [ ]:
def group_by_policy_and_sliding_window(df, window_size=3):
  """
  Groups data by policy and joins consecutive hours using a sliding window.

  Args:
    df: Pandas DataFrame with 'POLICY', 'WEEKDAY', 'HOUR', and 'CONSUMPTION' columns.
    window_size: Size of the sliding window (number of consecutive hours to join).

  Returns:
    A new DataFrame with grouped and joined data.
  """

  def join_consecutive_hours(group):
    """
    Joins consecutive hours within each group using a sliding window.
    """
    joined_data = []
    for i in range(len(group) - window_size + 1):
      window = group.iloc[i : i + window_size]
      joined_row = {
          'POLICY': window['POLICY'].iloc[0],
          'WEEKDAY': window['WEEKDAY'].iloc[0],
          'HOUR_START': window['HOUR'].iloc[0],
          'CONSUMPTION_1': window['CONSUMPTION'].iloc[0],
          'CONSUMPTION_2': window['CONSUMPTION'].iloc[1],
          'CONSUMPTION_3': window['CONSUMPTION'].iloc[2],
          # Add more CONSUMPTION_x columns as needed based on window_size
      }
      joined_data.append(joined_row)
    return pd.DataFrame(joined_data)

  # Group by 'POLICY'
  grouped = df.groupby('POLICY')

  # Apply the join_consecutive_hours function to each group
  joined_df = grouped.apply(join_consecutive_hours).reset_index(drop=True)

  return joined_df

# Assuming you have a DataFrame 'df' with the necessary columns

# Group and join data
# joined_df = group_by_policy_and_sliding_window(filtered_df, window_size=3)

# Print the joined DataFrame
# new_filtered_df.mean()

In [ ]:
average = new_filtered_df.groupby(['POLICY', 'USAGE', 'HOUSING', 'WEEKDAY', 'HOUR'])['FLOW'].mean().reset_index()

In [ ]:
average

In [ ]:
def join_hourly_averages_in_triplets(average_df):
  """
  Joins hourly average consumption data in triplets of consecutive hours.

  Args:
    average_df: Pandas DataFrame with 'POLICY', 'USAGE', 'HOUSING', 
                'WEEKDAY', 'HOUR', and 'FLOW' columns, representing 
                hourly average consumption.

  Returns:
    A new DataFrame with joined triplets of hourly averages.
  """

  def join_consecutive_hours(group):
    """
    Joins consecutive hours within each group into triplets.
    """
    joined_data = []
    for i in range(len(group) - 3):  # Adjust for triplets
      window = group.iloc[i : i + 4]  # Window of 3 hours
      joined_row = {
          'USAGE': window['USAGE'].iloc[0],
          'HOUSING': window['HOUSING'].iloc[0],
          'WEEKDAY': window['WEEKDAY'].iloc[0],
          'HOUR_START': window['HOUR'].iloc[0],  # Starting hour of the triplet
          'FLOW_1': window['FLOW'].iloc[0],
          'FLOW_2': window['FLOW'].iloc[1],
          'FLOW_3': window['FLOW'].iloc[2],
          'FLOW_4': window['FLOW'].iloc[3],
      }
      joined_data.append(joined_row)
    return pd.DataFrame(joined_data)

  # Group by 'POLICY', 'USAGE', 'HOUSING', and 'WEEKDAY'
  grouped = average_df.groupby(['USAGE', 'HOUSING', 'WEEKDAY'])

  # Apply the join_consecutive_hours function to each group
  joined_df = grouped.apply(join_consecutive_hours).reset_index(drop=True)

  return joined_df

In [ ]:
joined_averages_df = join_hourly_averages_in_triplets(average)

In [ ]:
print(len(joined_averages_df['POLICY'].unique()))

In [ ]:
average_without_policy = new_filtered_df.groupby(['USAGE', 'HOUSING', 'WEEKDAY', 'HOUR'])['FLOW'].mean().reset_index()
average_without_policy

In [ ]:
joined_averages_df = join_hourly_averages_in_triplets(average_without_policy)

In [ ]:
joined_averages_df

In [ ]:
import pandas as pd
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler  


def label_with_gaussian_mixture(df, n_components=2, threshold=0.5):
    """
    Applies Gaussian Mixture Model (GMM) to label data based on probabilities.

    Args:
      df: Pandas DataFrame with features for clustering (e.g., 'FLOW_1', 'FLOW_2', 'FLOW_3').
      n_components: Number of components (clusters) for the GMM.
      threshold: Probability threshold for assigning a label.

    Returns:
      Pandas DataFrame with an additional 'LEAK' column (True for potential leak, False otherwise).
    """

    # Select features for clustering
    X = df[['FLOW_1', 'FLOW_2', 'FLOW_3', 'FLOW_4']]

    # Standardize features
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # Apply Gaussian Mixture Model
    gmm = GaussianMixture(n_components=n_components, random_state=42)
    gmm.fit(X_scaled)

    # Get probabilities of belonging to each cluster
    probabilities = gmm.predict_proba(X_scaled)

    # Assign labels based on probability threshold
    df['LEAK'] = probabilities[:, 1] > threshold  # Assuming cluster 1 represents potential leaks

    return df

# Assuming you have the 'joined_averages_df' from the previous step

# Apply Gaussian Mixture for labeling
labeled_df = label_with_gaussian_mixture(joined_averages_df, n_components=2, threshold=0.5)



In [ ]:
# Print the labeled DataFrame
labeled_df